# 06 Train Metric Classifier

Train a small classifier on deterministic comparison metrics.

Input is not raw code. The production flow should be:

```text
suggestion + merged diff/code -> deterministic metrics -> classifier -> 0% / partial / mostly / 100%
```

This notebook trains on both the internal hand-labeled dataset and the LLM-reviewed Hugging Face imported dataset, then evaluates performance separately by source.


In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
INTERNAL_DATASET_DIR = PROJECT_ROOT / 'data' / 'processed' / 'pr_suggestion_coverage' / 'dataset'
INTERNAL_SCORES_PATH = PROJECT_ROOT / 'reports' / 'metric_scores.csv'
HF_DATASET_DIR = PROJECT_ROOT / 'data' / 'external' / 'github_codereview' / 'dataset'
HF_SCORES_PATH = PROJECT_ROOT / 'data' / 'external' / 'github_codereview' / 'metric_scores.csv'
MODEL_DIR = PROJECT_ROOT / 'models' / 'pr_suggestion_coverage'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

LABEL_ORDER = ['0%', 'partial', 'mostly', '100%']
LABEL_TO_PERCENTAGE = {'0%': 0, 'partial': 40, 'mostly': 80, '100%': 100}
INTERNAL_SAMPLE_WEIGHT = 1.0
HF_SAMPLE_WEIGHT = 1.0  # reduce to 0.2-0.5 if HF labels become noisy again
RANDOM_STATE = 42

MODEL_DIR


## Load Scores

`metric_scores.csv` contains features and rule-based predictions. We join `labels.csv` only for repo/source metadata used in grouped splits.


In [ ]:
def read_labels(dataset_dir: Path) -> pd.DataFrame:
    labels = pd.read_csv(dataset_dir / 'labels.csv')
    keep_columns = ['example_id', 'pr_url', 'repo', 'suggestion_source']
    return labels[[column for column in keep_columns if column in labels.columns]].copy()


def load_source_scores(source_name: str, scores_path: Path, dataset_dir: Path, sample_weight: float) -> pd.DataFrame:
    scores = pd.read_csv(scores_path)
    labels = read_labels(dataset_dir)
    rows = scores.merge(labels, on='example_id', how='left', suffixes=('', '_label_file'))
    rows['dataset_source'] = source_name
    rows['sample_weight'] = sample_weight
    rows['repo'] = rows['repo'].fillna(source_name + '/unknown')
    rows['pr_url'] = rows['pr_url'].fillna(rows['repo'].astype(str) + '#unknown-pr')
    rows['group_id'] = rows['dataset_source'] + ':' + rows['pr_url'].astype(str)
    return rows


internal_scores = load_source_scores('internal', INTERNAL_SCORES_PATH, INTERNAL_DATASET_DIR, INTERNAL_SAMPLE_WEIGHT)
hf_scores = load_source_scores('hf_github_codereview', HF_SCORES_PATH, HF_DATASET_DIR, HF_SAMPLE_WEIGHT)
scores = pd.concat([internal_scores, hf_scores], ignore_index=True)

print('rows:', len(scores))
print(scores['dataset_source'].value_counts())
print(scores['label'].value_counts().reindex(LABEL_ORDER).fillna(0).astype(int))
scores.head()


## Feature Schema

These are production-safe features: they require only suggestion + merged diff/code.


In [ ]:
NUMERIC_FEATURES = [
    'line_recall',
    'token_recall',
    'identifier_normalized_token_recall',
    'literal_normalized_token_recall',
    'identifier_and_literal_normalized_token_recall',
    'best_added_line_overlap',
    'best_hunk_token_recall',
    'best_hunk_token_precision',
    'best_hunk_token_f1',
    'best_hunk_identifier_normalized_recall',
    'best_hunk_literal_normalized_recall',
    'best_hunk_identifier_and_literal_normalized_recall',
    'best_hunk_contiguous_line_ratio',
    'best_hunk_token_lcs_recall',
    'best_hunk_size_ratio',
    'meaningful_anchor_recall',
    'meaningful_anchor_count',
    'best_hunk_size',
    'candidate_hunk_count',
    'structural_similarity',
    'structural_node_recall',
    'gumtree_operation_count',
    'gumtree_insert_ratio',
    'gumtree_delete_ratio',
    'gumtree_update_ratio',
    'gumtree_move_ratio',
    'file_overlap_ratio',
    'changed_line_overlap_ratio',
]
BOOLEAN_FEATURES = [
    'exact_normalized_match',
    'structural_available',
    'gumtree_available',
]
CATEGORICAL_FEATURES = [
    'suggestion_language',
    'tokenizer',
    'best_hunk_candidate_type',
    'structural_engine',
    'structural_language',
]

missing_features = [feature for feature in NUMERIC_FEATURES + BOOLEAN_FEATURES + CATEGORICAL_FEATURES if feature not in scores.columns]
if missing_features:
    raise ValueError(f'Missing features: {missing_features}')

for feature in NUMERIC_FEATURES:
    scores[feature] = pd.to_numeric(scores[feature], errors='coerce')
for feature in BOOLEAN_FEATURES:
    scores[feature] = scores[feature].astype(str).str.lower().eq('true').astype(int)
for feature in CATEGORICAL_FEATURES:
    scores[feature] = scores[feature].fillna('none').replace('', 'none').astype(str)

FEATURE_COLUMNS = NUMERIC_FEATURES + BOOLEAN_FEATURES + CATEGORICAL_FEATURES
X = scores[FEATURE_COLUMNS]
y = scores['label'].astype(str)
groups = scores['group_id'].astype(str)
sample_weights = scores['sample_weight'].astype(float)
X.shape, y.value_counts().reindex(LABEL_ORDER).fillna(0).astype(int)


## Grouped Split

Split by source+repo to reduce leakage from the same repository appearing in both train and test.


In [ ]:
def source_aware_group_split(rows: pd.DataFrame, test_size: float = 0.25) -> tuple[np.ndarray, np.ndarray]:
    train_parts: list[np.ndarray] = []
    test_parts: list[np.ndarray] = []

    for _, source_rows in rows.groupby('dataset_source', sort=False):
        source_indices = source_rows.index.to_numpy()
        source_groups = source_rows['group_id'].astype(str)

        if source_groups.nunique() < 2 or len(source_rows) < 4:
            fallback_split = max(1, int(round(len(source_rows) * (1 - test_size))))
            train_parts.append(source_indices[:fallback_split])
            test_parts.append(source_indices[fallback_split:])
            continue

        source_splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=RANDOM_STATE)
        train_positions, test_positions = next(
            source_splitter.split(source_rows[FEATURE_COLUMNS], source_rows['label'], groups=source_groups)
        )
        train_parts.append(source_indices[train_positions])
        test_parts.append(source_indices[test_positions])

    return np.concatenate(train_parts), np.concatenate(test_parts)


train_index, test_index = source_aware_group_split(scores, test_size=0.25)

X_train = X.loc[train_index]
X_test = X.loc[test_index]
y_train = y.loc[train_index]
y_test = y.loc[test_index]
weights_train = sample_weights.loc[train_index]
test_rows = scores.loc[test_index].copy()

print('train rows:', len(X_train))
print('test rows:', len(X_test))
print('train sources:')
print(scores.loc[train_index, 'dataset_source'].value_counts())
print('test sources:')
print(test_rows['dataset_source'].value_counts())
print('test labels:')
print(y_test.value_counts().reindex(LABEL_ORDER).fillna(0).astype(int))
print('test PR groups by source:')
print(test_rows.groupby('dataset_source')['group_id'].nunique())


## Train Candidate Models


In [ ]:
numeric_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='none')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_preprocessor, NUMERIC_FEATURES + BOOLEAN_FEATURES),
    ('categorical', categorical_preprocessor, CATEGORICAL_FEATURES),
])

models = {
    'logistic_regression': LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE),
    'random_forest': RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=3,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
    ),
    'hist_gradient_boosting': HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.05,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    ),
}

trained_models = {}
for name, estimator in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', estimator),
    ])
    pipeline.fit(X_train, y_train, classifier__sample_weight=weights_train)
    trained_models[name] = pipeline

list(trained_models)


## Evaluate


In [ ]:
def percentage_mae(labels: pd.Series, predictions: np.ndarray) -> float:
    actual_percentages = labels.map(LABEL_TO_PERCENTAGE).to_numpy()
    predicted_percentages = np.array([LABEL_TO_PERCENTAGE[prediction] for prediction in predictions])
    return float(mean_absolute_error(actual_percentages, predicted_percentages))


def evaluate_predictions(labels: pd.Series, predictions: np.ndarray) -> dict:
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'macro_f1': float(f1_score(labels, predictions, labels=LABEL_ORDER, average='macro', zero_division=0)),
        'percentage_mae': percentage_mae(labels, predictions),
        'confusion_matrix': confusion_matrix(labels, predictions, labels=LABEL_ORDER).tolist(),
        'classification_report': classification_report(labels, predictions, labels=LABEL_ORDER, zero_division=0, output_dict=True),
    }


evaluation_rows = []
evaluation_report = {'label_order': LABEL_ORDER, 'models': {}}
rule_predictions = test_rows['predicted_label'].astype(str).to_numpy()
rule_metrics = evaluate_predictions(y_test, rule_predictions)
evaluation_report['models']['rule_based_current'] = rule_metrics
evaluation_rows.append({'model': 'rule_based_current', **{key: rule_metrics[key] for key in ['accuracy', 'macro_f1', 'percentage_mae']}})

for name, model in trained_models.items():
    predictions = model.predict(X_test)
    metrics = evaluate_predictions(y_test, predictions)
    evaluation_report['models'][name] = metrics
    evaluation_rows.append({'model': name, **{key: metrics[key] for key in ['accuracy', 'macro_f1', 'percentage_mae']}})

evaluation = pd.DataFrame(evaluation_rows).sort_values(['macro_f1', 'accuracy'], ascending=False)
evaluation


## Per-Source Evaluation


In [ ]:
best_model_name = evaluation.iloc[0]['model']
best_model = None if best_model_name == 'rule_based_current' else trained_models[best_model_name]

per_source_rows = []
for source_name, source_rows in test_rows.groupby('dataset_source'):
    source_labels = source_rows['label'].astype(str)
    if best_model is None:
        source_predictions = source_rows['predicted_label'].astype(str).to_numpy()
    else:
        source_predictions = best_model.predict(source_rows[FEATURE_COLUMNS])
    source_metrics = evaluate_predictions(source_labels, source_predictions)
    per_source_rows.append({
        'dataset_source': source_name,
        'rows': len(source_rows),
        'accuracy': source_metrics['accuracy'],
        'macro_f1': source_metrics['macro_f1'],
        'percentage_mae': source_metrics['percentage_mae'],
    })

per_source_evaluation = pd.DataFrame(per_source_rows).sort_values('dataset_source')
per_source_evaluation


## Confusion Matrix And Failure Rows


In [ ]:
if best_model is None:
    best_predictions = test_rows['predicted_label'].astype(str).to_numpy()
else:
    best_predictions = best_model.predict(X_test)

confusion = pd.DataFrame(
    confusion_matrix(y_test, best_predictions, labels=LABEL_ORDER),
    index=[f'actual_{label}' for label in LABEL_ORDER],
    columns=[f'pred_{label}' for label in LABEL_ORDER],
)
display(confusion)

failure_rows = test_rows.copy()
failure_rows['model_prediction'] = best_predictions
failure_rows['model_correct'] = failure_rows['label'] == failure_rows['model_prediction']
failure_rows.loc[~failure_rows['model_correct'], [
    'example_id',
    'dataset_source',
    'repo',
    'label',
    'predicted_label',
    'model_prediction',
    'suggestion_language',
    'best_hunk_token_f1',
    'best_hunk_contiguous_line_ratio',
    'best_hunk_size_ratio',
    'meaningful_anchor_recall',
    'changed_line_overlap_ratio',
]].head(25)


## Save Best Model

If the rule-based predictor wins, this notebook still saves the best trained model separately and records the rule baseline in the report.


In [ ]:
trained_evaluation = evaluation[evaluation['model'] != 'rule_based_current'].copy()
best_trained_model_name = trained_evaluation.iloc[0]['model']
best_trained_model = trained_models[best_trained_model_name]

model_path = MODEL_DIR / 'model.joblib'
schema_path = MODEL_DIR / 'feature_schema.json'
report_path = MODEL_DIR / 'evaluation_report.json'

joblib.dump(best_trained_model, model_path)
feature_schema = {
    'model_name': best_trained_model_name,
    'label_order': LABEL_ORDER,
    'numeric_features': NUMERIC_FEATURES,
    'boolean_features': BOOLEAN_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'feature_columns': FEATURE_COLUMNS,
    'prediction_input': 'deterministic metric row derived from suggestion + merged diff/code',
}
schema_path.write_text(json.dumps(feature_schema, indent=2), encoding='utf-8')

evaluation_report['best_overall_by_macro_f1'] = str(best_model_name)
evaluation_report['saved_trained_model'] = str(best_trained_model_name)
evaluation_report['evaluation_table'] = evaluation.to_dict(orient='records')
evaluation_report['per_source_evaluation'] = per_source_evaluation.to_dict(orient='records')
report_path.write_text(json.dumps(evaluation_report, indent=2), encoding='utf-8')

model_path, schema_path, report_path
